<a href="https://colab.research.google.com/github/ReversePI78/-DevelRocket-AI-game-review-analysis/blob/main/11%EC%B0%A8%EC%8B%9C_%EC%A0%84%EC%B2%98%EB%A6%AC%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8%EA%B5%AC%EC%B6%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11차시 프로젝트: 전처리 파이프라인 구축

**재사용 가능한 텍스트 전처리 시스템 만들기**

---

## 프로젝트 개요

### 학습 목표
- 재사용 가능한 텍스트 전처리 파이프라인을 구축할 수 있다
- 전처리 단계를 모듈화하여 관리할 수 있다
- 각 전처리 단계를 테스트하고 검증할 수 있다

### 왜 파이프라인이 필요할까요?

**문제 상황**: 전처리 코드를 매번 복사-붙여넣기하면...
- 코드가 중복되어 수정이 어려움
- 어떤 전처리를 했는지 추적 어려움
- 실수로 단계를 건너뛸 수 있음

**해결책**: 파이프라인으로 만들면...
- 한 번 만들어두면 계속 재사용
- 전처리 과정이 명확히 기록됨
- 순서가 보장되어 실수 방지

---

## 1단계: 필요한 라이브러리 설치 및 불러오기

먼저 전처리에 필요한 도구들을 준비합니다.

!pip install pandas numpy

In [1]:
# 필요한 라이브러리 설치 (Colab에서 처음 실행 시)
# !pip install pandas numpy

import pandas as pd
import numpy as np
import re
from typing import Callable, List

print("✅ 라이브러리 불러오기 완료")

✅ 라이브러리 불러오기 완료


## 2단계: 샘플 데이터 준비

실제 Steam 리뷰와 비슷한 샘플 데이터를 만들어 테스트해봅니다.

**실제 데이터 특징**:
- 불필요한 공백이 많음
- 특수문자가 섞여 있음
- 이모지가 포함됨
- 대소문자가 섞여 있음

In [2]:
# 샘플 리뷰 데이터 (실제와 유사하게 노이즈 포함)
sample_reviews = [
    "   This game is AMAZING!!!   ",  # 앞뒤 공백, 대문자, 느낌표
    "Great game 😊 but laggy sometimes...",  # 이모지, 특수문자
    "I love this game!!!!! 10/10",  # 연속 느낌표, 슬래시
    "Not recommended... too many bugs 🐛",  # 이모지, 점 연속
    "BEST GAME EVER!!! <3 <3 <3",  # 대문자, 하트 이모티콘
]

# DataFrame으로 변환
df = pd.DataFrame({'review_text': sample_reviews})

print("원본 리뷰 데이터:")
print(df)
print(f"\n데이터 크기: {len(df)}개")

원본 리뷰 데이터:
                           review_text
0           This game is AMAZING!!!   
1  Great game 😊 but laggy sometimes...
2          I love this game!!!!! 10/10
3   Not recommended... too many bugs 🐛
4           BEST GAME EVER!!! <3 <3 <3

데이터 크기: 5개


## 3단계: 전처리 함수 만들기 (모듈화)

각 전처리 작업을 **독립적인 함수**로 만듭니다.

### 왜 함수로 만들까요?
- 각 단계를 독립적으로 테스트 가능
- 필요한 단계만 선택해서 사용 가능
- 나중에 수정하기 쉬움

---

### 함수 1: 공백 제거

In [3]:
def remove_extra_whitespace(text: str) -> str:
    """
    앞뒤 공백과 중간의 연속된 공백을 제거합니다.

    예시:
    "   hello    world   " → "hello world"
    """
    # 1. 앞뒤 공백 제거
    text = text.strip()

    # 2. 중간의 연속된 공백을 하나로
    text = re.sub(r'\s+', ' ', text)

    return text

# 테스트
test_text = "   hello    world   "
result = remove_extra_whitespace(test_text)
print(f"원본: '{test_text}'")
print(f"결과: '{result}'")
print("✅ 공백 제거 함수 작동 확인")

원본: '   hello    world   '
결과: 'hello world'
✅ 공백 제거 함수 작동 확인


### 함수 2: 소문자 변환

In [4]:
def to_lowercase(text: str) -> str:
    """
    모든 문자를 소문자로 변환합니다.

    예시:
    "HELLO World" → "hello world"
    """
    return text.lower()

# 테스트
test_text = "HELLO World"
result = to_lowercase(test_text)
print(f"원본: '{test_text}'")
print(f"결과: '{result}'")
print("✅ 소문자 변환 함수 작동 확인")

원본: 'HELLO World'
결과: 'hello world'
✅ 소문자 변환 함수 작동 확인


### 함수 3: 특수문자 제거

In [5]:
def remove_special_chars(text: str) -> str:
    """
    영문, 숫자, 공백을 제외한 모든 문자를 제거합니다.

    예시:
    "hello!!! world..." → "hello world"
    """
    # [^...] 는 "...가 아닌 것"을 의미
    # a-zA-Z: 영문, 0-9: 숫자, \s: 공백
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    return text

# 테스트
test_text = "hello!!! world... 😊"
result = remove_special_chars(test_text)
print(f"원본: '{test_text}'")
print(f"결과: '{result}'")
print("✅ 특수문자 제거 함수 작동 확인")

원본: 'hello!!! world... 😊'
결과: 'hello world '
✅ 특수문자 제거 함수 작동 확인


### 함수 4: 연속된 문자 제거

In [6]:
def remove_repeated_chars(text: str, max_repeat: int = 2) -> str:
    """
    연속된 같은 문자를 제한합니다.

    예시:
    "hellooooo" → "helloo" (max_repeat=2인 경우)
    """
    # (.)는 아무 문자 하나, \1은 그 문자를 반복
    # {2,}는 2번 이상 반복
    pattern = r'(.)\1{' + str(max_repeat) + r',}'
    replacement = r'\1' * max_repeat

    return re.sub(pattern, replacement, text)

# 테스트
test_text = "hellooooo woooorld"
result = remove_repeated_chars(test_text, max_repeat=2)
print(f"원본: '{test_text}'")
print(f"결과: '{result}'")
print("✅ 연속 문자 제거 함수 작동 확인")

원본: 'hellooooo woooorld'
결과: 'helloo woorld'
✅ 연속 문자 제거 함수 작동 확인


## 4단계: 파이프라인 클래스 만들기

여러 전처리 함수를 **순서대로 자동 실행**하는 파이프라인을 만듭니다.

### 파이프라인의 장점
1. 전처리 순서가 자동으로 지켜짐
2. 함수를 자유롭게 추가/제거 가능
3. 전체 DataFrame에 한번에 적용 가능

In [7]:
class TextPreprocessingPipeline:
    """
    텍스트 전처리 파이프라인

    사용법:
    1. 파이프라인 생성
    2. 전처리 함수들을 순서대로 추가
    3. 데이터에 적용
    """

    def __init__(self):
        """파이프라인 초기화"""
        self.steps = []  # 전처리 함수들을 담을 리스트

    def add_step(self, func: Callable[[str], str], name: str = None):
        """
        전처리 단계를 추가합니다.

        Args:
            func: 전처리 함수
            name: 단계 이름 (선택사항)
        """
        if name is None:
            name = func.__name__  # 함수 이름을 자동으로 사용

        self.steps.append({'name': name, 'func': func})
        print(f"✅ 단계 추가: {name}")
        return self  # 메서드 체이닝을 위해 self 반환

    def process(self, text: str, verbose: bool = False) -> str:
        """
        하나의 텍스트에 파이프라인을 적용합니다.

        Args:
            text: 원본 텍스트
            verbose: True면 각 단계별 결과 출력

        Returns:
            전처리된 텍스트
        """
        result = text

        for step in self.steps:
            result = step['func'](result)

            if verbose:
                print(f"{step['name']:20s} → {result}")

        return result

    def process_dataframe(self, df: pd.DataFrame, column: str,
                          output_column: str = None) -> pd.DataFrame:
        """
        DataFrame의 텍스트 컬럼에 파이프라인을 적용합니다.

        Args:
            df: 데이터프레임
            column: 전처리할 컬럼 이름
            output_column: 결과를 저장할 컬럼 이름 (None이면 원본 덮어쓰기)

        Returns:
            전처리된 데이터프레임
        """
        if output_column is None:
            output_column = column

        print(f"📝 {len(df)}개 텍스트 전처리 중...")

        # apply로 각 행에 process 적용
        df[output_column] = df[column].apply(lambda x: self.process(x, verbose=False))

        print("✅ 전처리 완료!")
        return df

    def get_steps(self) -> List[str]:
        """현재 등록된 전처리 단계 목록을 반환합니다."""
        return [step['name'] for step in self.steps]

print("✅ TextPreprocessingPipeline 클래스 정의 완료")

✅ TextPreprocessingPipeline 클래스 정의 완료


## 5단계: 파이프라인 구축 및 테스트

이제 만든 함수들을 파이프라인에 추가하고 테스트해봅니다.

In [8]:
# 파이프라인 생성
pipeline = TextPreprocessingPipeline()

# 전처리 단계를 순서대로 추가
pipeline.add_step(remove_extra_whitespace, "공백 제거")
pipeline.add_step(to_lowercase, "소문자 변환")
pipeline.add_step(remove_special_chars, "특수문자 제거")
pipeline.add_step(remove_repeated_chars, "연속 문자 제거")

print("\n현재 파이프라인 단계:")
for i, step in enumerate(pipeline.get_steps(), 1):
    print(f"  {i}. {step}")

✅ 단계 추가: 공백 제거
✅ 단계 추가: 소문자 변환
✅ 단계 추가: 특수문자 제거
✅ 단계 추가: 연속 문자 제거

현재 파이프라인 단계:
  1. 공백 제거
  2. 소문자 변환
  3. 특수문자 제거
  4. 연속 문자 제거


### 단일 텍스트 테스트 (verbose 모드)

In [9]:
# 테스트용 복잡한 텍스트
test_text = "   This game is AMAZING!!!!! 😊 I love it!!!   "

print("원본:")
print(f"  '{test_text}'")
print("\n각 단계별 결과:")

result = pipeline.process(test_text, verbose=True)

print("\n최종 결과:")
print(f"  '{result}'")

원본:
  '   This game is AMAZING!!!!! 😊 I love it!!!   '

각 단계별 결과:
공백 제거                → This game is AMAZING!!!!! 😊 I love it!!!
소문자 변환               → this game is amazing!!!!! 😊 i love it!!!
특수문자 제거              → this game is amazing  i love it
연속 문자 제거             → this game is amazing  i love it

최종 결과:
  'this game is amazing  i love it'


## 6단계: DataFrame에 파이프라인 적용

샘플 리뷰 데이터 전체에 파이프라인을 적용해봅니다.

In [10]:
# 원본 데이터 확인
print("=== 전처리 전 ===")
print(df)
print()

=== 전처리 전 ===
                           review_text
0           This game is AMAZING!!!   
1  Great game 😊 but laggy sometimes...
2          I love this game!!!!! 10/10
3   Not recommended... too many bugs 🐛
4           BEST GAME EVER!!! <3 <3 <3



In [11]:
# 파이프라인 적용 (새 컬럼에 저장)
df_processed = pipeline.process_dataframe(
    df.copy(),  # 원본 보존을 위해 복사본 사용
    column='review_text',
    output_column='processed_text'
)

print("\n=== 전처리 후 ===")
print(df_processed)

📝 5개 텍스트 전처리 중...
✅ 전처리 완료!

=== 전처리 후 ===
                           review_text                   processed_text
0           This game is AMAZING!!!                this game is amazing
1  Great game 😊 but laggy sometimes...  great game  but laggy sometimes
2          I love this game!!!!! 10/10            i love this game 1010
3   Not recommended... too many bugs 🐛   not recommended too many bugs 
4           BEST GAME EVER!!! <3 <3 <3             best game ever 3 3 3


### 전후 비교

In [12]:
# 전후 비교를 위한 시각화
print("\n=== 전처리 전후 비교 ===")
print("="*80)

for idx, row in df_processed.iterrows():
    print(f"\n[리뷰 {idx+1}]")
    print(f"  전: '{row['review_text']}'")
    print(f"  후: '{row['processed_text']}'")


=== 전처리 전후 비교 ===

[리뷰 1]
  전: '   This game is AMAZING!!!   '
  후: 'this game is amazing'

[리뷰 2]
  전: 'Great game 😊 but laggy sometimes...'
  후: 'great game  but laggy sometimes'

[리뷰 3]
  전: 'I love this game!!!!! 10/10'
  후: 'i love this game 1010'

[리뷰 4]
  전: 'Not recommended... too many bugs 🐛'
  후: 'not recommended too many bugs '

[리뷰 5]
  전: 'BEST GAME EVER!!! <3 <3 <3'
  후: 'best game ever 3 3 3'


## 7단계: 파이프라인 검증

전처리가 제대로 되었는지 확인하는 검증 함수를 만들어봅니다.

In [13]:
def validate_preprocessing(df: pd.DataFrame, column: str):
    """
    전처리된 텍스트가 올바른지 검증합니다.

    검증 항목:
    1. 앞뒤 공백이 없는지
    2. 대문자가 없는지
    3. 특수문자가 없는지
    """
    print("\n=== 전처리 검증 ===")
    issues = []

    for idx, text in enumerate(df[column]):
        # 1. 앞뒤 공백 체크
        if text != text.strip():
            issues.append(f"리뷰 {idx}: 앞뒤 공백 존재")

        # 2. 대문자 체크
        if text != text.lower():
            issues.append(f"리뷰 {idx}: 대문자 존재")

        # 3. 특수문자 체크 (영문, 숫자, 공백 외)
        if re.search(r'[^a-zA-Z0-9\s]', text):
            issues.append(f"리뷰 {idx}: 특수문자 존재")

    if issues:
        print("❌ 전처리 문제 발견:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("✅ 전처리 완벽! 모든 검증 통과")

    return len(issues) == 0

# 검증 실행
is_valid = validate_preprocessing(df_processed, 'processed_text')


=== 전처리 검증 ===
❌ 전처리 문제 발견:
  - 리뷰 3: 앞뒤 공백 존재


## 8단계: 실전 활용 - 실제 데이터에 적용

이제 여러분이 수집한 Steam 리뷰 데이터에 파이프라인을 적용해보세요!

### 작업 순서
1. 여러분의 리뷰 데이터 불러오기
2. 파이프라인 적용
3. 결과 저장

In [14]:
# 실습: 여러분의 데이터 경로를 입력하세요
# 예: my_reviews = pd.read_csv('steam_reviews.csv')

# TODO: 데이터 불러오기
# my_reviews = pd.read_csv('YOUR_FILE.csv')

# TODO: 파이프라인 적용
# my_reviews_processed = pipeline.process_dataframe(
#     my_reviews,
#     column='review',  # 텍스트가 있는 컬럼명
#     output_column='review_processed'
# )

# TODO: 결과 확인
# print(my_reviews_processed[['review', 'review_processed']].head())

# TODO: 검증
# validate_preprocessing(my_reviews_processed, 'review_processed')

# TODO: 저장
# my_reviews_processed.to_csv('steam_reviews_processed.csv', index=False)

print("💡 위 코드의 주석을 해제하고 여러분의 데이터로 실습해보세요!")

💡 위 코드의 주석을 해제하고 여러분의 데이터로 실습해보세요!


## 9단계: 추가 전처리 함수 만들기 (선택)

필요에 따라 더 많은 전처리 함수를 만들 수 있습니다.

### 예시: URL 제거

In [15]:
def remove_urls(text: str) -> str:
    """
    URL을 제거합니다.

    예시:
    "Check out https://example.com" → "Check out"
    """
    # http:// 또는 https://로 시작하는 URL 제거
    text = re.sub(r'https?://\S+', '', text)
    return text

# 테스트
test_text = "Great game! Visit https://example.com for more info"
result = remove_urls(test_text)
print(f"원본: {test_text}")
print(f"결과: {result}")

원본: Great game! Visit https://example.com for more info
결과: Great game! Visit  for more info


### 예시: 짧은 텍스트 필터링

In [16]:
def filter_short_text(df: pd.DataFrame, column: str, min_length: int = 10) -> pd.DataFrame:
    """
    너무 짧은 텍스트를 제거합니다.

    Args:
        df: 데이터프레임
        column: 텍스트 컬럼
        min_length: 최소 글자 수
    """
    before = len(df)
    df_filtered = df[df[column].str.len() >= min_length]
    after = len(df_filtered)

    print(f"전: {before}개 → 후: {after}개 (제거: {before-after}개)")
    return df_filtered

# 테스트
df_filtered = filter_short_text(df_processed.copy(), 'processed_text', min_length=15)
print("\n남은 리뷰:")
print(df_filtered)

전: 5개 → 후: 5개 (제거: 0개)

남은 리뷰:
                           review_text                   processed_text
0           This game is AMAZING!!!                this game is amazing
1  Great game 😊 but laggy sometimes...  great game  but laggy sometimes
2          I love this game!!!!! 10/10            i love this game 1010
3   Not recommended... too many bugs 🐛   not recommended too many bugs 
4           BEST GAME EVER!!! <3 <3 <3             best game ever 3 3 3


## 10단계: 내 Steam 리뷰(한국어)에 파이프라인 적용

8차시에 수집한 스타듀 밸리 한국어 리뷰(36,214건)에 파이프라인을 적용합니다. CSV는 왼쪽 파일 패널로 업로드했습니다.

**보완한 점**
- 기본 `remove_special_chars`는 영문·숫자만 남겨서 **한글이 모두 지워짐** → 한글(가-힣, ㄱ-ㅎ, ㅏ-ㅣ)을 남기는 함수를 추가
- 특수문자를 빈 문자열로 지우면 단어가 붙거나(예: 10/10 → 1010) 공백이 남음 → 공백으로 바꾸고, 마지막에 공백 정리를 한 번 더 실행
- 7단계 검증의 '앞뒤 공백 존재'도 같은 원인(공백 정리가 특수문자 제거보다 먼저 실행됨)
- 실제 리뷰에 있는 URL을 지우는 9단계 `remove_urls`도 파이프라인에 넣음

In [17]:
# 한국어 리뷰용 특수문자 제거: 한글·영문·숫자·공백만 남기고, 나머지는 공백으로 바꿈
def remove_special_chars_ko(text: str) -> str: return re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', text)

sample = '정말 재밌어요!! ㅋㅋㅋ 10/10 😊'
print('원본      :', sample)
print('기본 함수 :', repr(remove_special_chars(sample)))
print('한국어 함수:', repr(remove_special_chars_ko(sample)))

원본      : 정말 재밌어요!! ㅋㅋㅋ 10/10 😊
기본 함수 : '   1010 '
한국어 함수: '정말 재밌어요   ㅋㅋㅋ 10 10  '


In [18]:
# 한국어 파이프라인: URL 제거를 앞에 넣고, 특수문자 제거 뒤 공백을 한 번 더 정리
pipeline_ko = TextPreprocessingPipeline()
pipeline_ko.add_step(remove_urls, 'URL 제거')
pipeline_ko.add_step(remove_extra_whitespace, '공백 제거')
pipeline_ko.add_step(to_lowercase, '소문자 변환')
pipeline_ko.add_step(remove_special_chars_ko, '특수문자 제거(한글 유지)')
pipeline_ko.add_step(remove_repeated_chars, '연속 문자 제거')
pipeline_ko.add_step(remove_extra_whitespace, '공백 재정리')
print()
_ = pipeline_ko.process('   스타듀밸리 진짜 최고!!!!! ㅋㅋㅋㅋㅋ 😊 https://store.steampowered.com  GOOD game   ', verbose=True)

✅ 단계 추가: URL 제거
✅ 단계 추가: 공백 제거
✅ 단계 추가: 소문자 변환
✅ 단계 추가: 특수문자 제거(한글 유지)
✅ 단계 추가: 연속 문자 제거
✅ 단계 추가: 공백 재정리

URL 제거               →    스타듀밸리 진짜 최고!!!!! ㅋㅋㅋㅋㅋ 😊   GOOD game   
공백 제거                → 스타듀밸리 진짜 최고!!!!! ㅋㅋㅋㅋㅋ 😊 GOOD game
소문자 변환               → 스타듀밸리 진짜 최고!!!!! ㅋㅋㅋㅋㅋ 😊 good game
특수문자 제거(한글 유지)       → 스타듀밸리 진짜 최고      ㅋㅋㅋㅋㅋ   good game
연속 문자 제거             → 스타듀밸리 진짜 최고  ㅋㅋ  good game
공백 재정리               → 스타듀밸리 진짜 최고 ㅋㅋ good game


In [19]:
# 8차시에 수집한 리뷰 불러오기 + 파이프라인 적용
mine = pd.read_csv('Stardew_Valley_reviews_20261122.csv')
print('불러온 리뷰 수:', len(mine), '/ review_text 결측:', mine['review_text'].isnull().sum())
mine['review_text'] = mine['review_text'].fillna('').astype(str)
mine = pipeline_ko.process_dataframe(mine, column='review_text', output_column='review_processed')
mine[['review_text', 'review_processed']].head(10)

불러온 리뷰 수: 36214 / review_text 결측: 104
📝 36214개 텍스트 전처리 중...
✅ 전처리 완료!


,review_text,review_processed
0,레전드 힐링 게임,레전드 힐링 게임
1,시스템 디자인과 성장 구조가 정교하게 맞물린 명작.\r\n농사·채광·낚시 등 개별 ...,시스템 디자인과 성장 구조가 정교하게 맞물린 명작 농사 채광 낚시 등 개별 콘텐츠가...
2,그냥 하세요 제발 하세요 무조건 하세요 재미있으니까 하세요 지루한데 하세요 힐링 되...,그냥 하세요 제발 하세요 무조건 하세요 재미있으니까 하세요 지루한데 하세요 힐링 되...
3,존잼,존잼
4,재 밌 음,재 밌 음
5,문어발엔딩 잼남,문어발엔딩 잼남
6,160시간 중 단 1초도 시간이 아깝지 않앗음.\n\n모든 시간을 철저하게 영양가 ...,160시간 중 단 1초도 시간이 아깝지 않앗음 모든 시간을 철저하게 영양가 잇게 꽉...
7,히히 재밌다,히히 재밌다
8,여친이랑 잔잔하게 스몰토크 하면서 낚시할 때 제일 재밌었음,여친이랑 잔잔하게 스몰토크 하면서 낚시할 때 제일 재밌었음
9,다 아시잖아요,다 아시잖아요


In [20]:
# 검증: 한국어 기준으로 앞뒤 공백, 대문자, 특수문자, 빈 결과 확인 (7단계 검증 항목을 한국어용으로 바꾼 것)
s = mine['review_processed']
print('=== 한국어 전처리 검증 ===')
print('앞뒤 공백이 남은 리뷰 :', (s != s.str.strip()).sum())
print('대문자가 남은 리뷰   :', (s != s.str.lower()).sum())
print('특수문자가 남은 리뷰 :', s.str.contains(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', regex=True).sum())
print('연속 공백이 남은 리뷰 :', s.str.contains(r'\s{2,}', regex=True).sum())
print('전처리 후 빈 리뷰     :', (s.str.len() == 0).sum())
print('평균 글자 수          :', round(mine['review_text'].str.len().mean(), 1), '->', round(s.str.len().mean(), 1))
print('기본 함수로 처리했다면 한글이 사라져 빈 리뷰:', (pipeline.process_dataframe(mine[['review_text']].copy(), column='review_text')['review_text'].str.strip() == '').sum())

=== 한국어 전처리 검증 ===
앞뒤 공백이 남은 리뷰 : 0
대문자가 남은 리뷰   : 0
특수문자가 남은 리뷰 : 0
연속 공백이 남은 리뷰 : 0
전처리 후 빈 리뷰     : 819
평균 글자 수          : 41.2 -> 38.4
📝 36214개 텍스트 전처리 중...
✅ 전처리 완료!
기본 함수로 처리했다면 한글이 사라져 빈 리뷰: 29217


In [21]:
# 빈 결과가 된 리뷰와 짧은 리뷰 확인 후, 빈 리뷰만 제거하고 저장
# (한국어는 '갓겜'처럼 2글자 리뷰도 의미가 있어서 1글자 이상은 남김)
print('빈 결과가 된 원문 예시:', mine.loc[s.str.len() == 0, 'review_text'].head(8).tolist())
print('5글자 미만 리뷰 상위 10개:', mine.loc[s.str.len().between(1, 4), 'review_processed'].value_counts().head(10).to_dict())
mine_clean = filter_short_text(mine, 'review_processed', min_length=1)
mine_clean.to_csv('Stardew_Valley_reviews_processed.csv', index=False, encoding='utf-8-sig')
print('저장 완료: Stardew_Valley_reviews_processed.csv /', len(mine_clean), '개')

빈 결과가 된 원문 예시: ['', '爽完', '.', '.', ',', '.', '', '.']
5글자 미만 리뷰 상위 10개: {'good': 560, '갓겜': 513, '굿': 420, '재밌어요': 304, '재밌음': 259, '꿀잼': 206, '개꿀잼': 178, '존잼': 168, '굳': 168, '재밌다': 160}
전: 36214개 → 후: 35395개 (제거: 819개)
저장 완료: Stardew_Valley_reviews_processed.csv / 35395 개


## 11. 전처리 결과 메모

**[적용 결과]**
- 대상: 스타듀 밸리 한국어 리뷰 36,214건 (원문 결측 104건은 빈 문자열로 처리)
- 파이프라인 6단계: URL 제거 → 공백 제거 → 소문자 변환 → 특수문자 제거(한글 유지) → 연속 문자 제거 → 공백 재정리
- 검증: 앞뒤 공백·대문자·특수문자·연속 공백이 남은 리뷰 모두 0건
- 평균 글자 수: 41.2자 → 38.4자
- 전처리 후 빈 리뷰 819건을 제거하고 35,395건을 Stardew_Valley_reviews_processed.csv로 저장

**[발견한 점]**
1. **기본 함수는 한국어 데이터를 망가뜨린다**: 노트북 기본 특수문자 함수를 그대로 쓰면 36,214건 중 29,217건(약 81%)이 빈 리뷰가 된다. 한글 범위(가-힣, ㄱ-ㅎ, ㅏ-ㅣ)를 꼭 넣어야 한다.
2. **빈 결과 819건의 정체**: 원문 결측, 기호만 있는 리뷰('.', ','), 한국어 필터에 섞여 들어온 중국어 리뷰('爽完')였다.
3. **짧아도 뜻이 분명하다**: 5글자 미만 리뷰 상위는 'good'(560), '갓겜'(513), '굿'(420), '재밌어요'(304)다. 글자 수로 거르지 않고 빈 리뷰만 제거했다.
4. **단계 순서가 결과를 바꾼다**: 공백 정리를 특수문자 제거보다 먼저만 하면 지운 자리에 공백이 남는다(7단계 검증 실패 원인). 마지막에 공백 재정리 단계를 추가해 해결했다.

## 프로젝트 정리

### 오늘 만든 것

✅ **4가지 전처리 함수**
1. `remove_extra_whitespace` - 공백 제거
2. `to_lowercase` - 소문자 변환
3. `remove_special_chars` - 특수문자 제거
4. `remove_repeated_chars` - 연속 문자 제거

✅ **TextPreprocessingPipeline 클래스**
- 전처리 단계를 순서대로 관리
- 단일 텍스트/DataFrame 모두 처리 가능
- 각 단계별 결과 확인 가능 (verbose 모드)

✅ **검증 함수**
- 전처리가 올바르게 되었는지 자동 확인

---

### 핵심 개념

| 개념 | 설명 |
|------|------|
| **모듈화** | 각 기능을 독립적인 함수로 분리 |
| **파이프라인** | 여러 단계를 순서대로 자동 실행 |
| **재사용성** | 한 번 만들면 계속 사용 가능 |
| **검증** | 결과가 올바른지 확인하는 과정 |

